In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils import resample
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import random
from deap import base, creator, tools, algorithms

df = pd.read_csv("./df.csv")
df = df.rename(columns={ '孕妇代码':'patient_id', '孕妇BMI':'bmi', '检测孕周':'g_week', 'Y染色体浓度':'y_frac' })
df = df[['patient_id','bmi','g_week','y_frac']]

df['bmi'] = pd.to_numeric(df['bmi'])
df['g_week'] = pd.to_numeric(df['g_week'])
df['y_frac'] = pd.to_numeric(df['y_frac'])
THRES = 0.04
MIN_WEEK, MAX_WEEK = 11, 29
def estimate_cross_week(g, threshold=THRES, min_week=MIN_WEEK, max_week=MAX_WEEK):
    g = g.sort_values('g_week')
    x = g['g_week'].values.astype(float)
    y = g['y_frac'].values.astype(float)
    if len(x) == 0:
        return np.nan

    #  相邻点线性插值（若跨越阈值）
    for i in range(len(x)-1):
        y0, y1 = y[i], y[i+1]
        if (y0 < threshold and y1 > threshold) or (y0 > threshold and y1 < threshold):
            x0, x1 = x[i], x[i+1]
            if abs(y1 - y0) < 1e-12:
                continue
            root = x0 + (threshold - y0) * (x1 - x0) / (y1 - y0)
            if min_week <= root <= max_week:
                return float(root)

    #  二次拟合求根（样本>=3）
    if len(x) >= 3:
        try:
            coeff = np.polyfit(x, y, 2)  # a*x^2 + b*x + c
            coeff_mod = coeff.copy()
            coeff_mod[-1] -= threshold
            roots = np.roots(coeff_mod)
            real_roots = np.real(roots[np.isreal(roots)])
            cand = real_roots[(real_roots >= min_week) & (real_roots <= max_week)]
            if cand.size > 0:
                return float(np.min(cand))
        except Exception:
            pass
    #  向前线性外推（使用最早两个点）
    if len(x) >= 2:
        x0, x1 = x[0], x[1]
        y0, y1 = y[0], y[1]
        if (y1 - y0) > 1e-8 and abs(y1 - y0) > 1e-12:
            root = x0 + (threshold - y0) * (x1 - x0) / (y1 - y0)
            if min_week <= root <= x0:
                return float(root)
    meet_mask = y >= threshold
    if meet_mask.any():
        return float(np.min(x[meet_mask]))
    idx = int(np.argmin(np.abs(y - threshold)))
    candidate_week = x[idx]
    if min_week <= candidate_week <= max_week:
        return float(candidate_week)
    return np.nan

preds = df.groupby('patient_id').apply(lambda g: estimate_cross_week(g)).reset_index()
preds.columns = ['patient_id', 'pred_cross_week']
df1 = pd.DataFrame({'patient_id': df['patient_id'].unique()})
df1 = df1.merge(preds, on='patient_id', how='left')
# 提取每个孕妇的平均BMI
patient_bmi = df.groupby('patient_id')['bmi'].mean().reset_index()
# 将BMI信息合并到df1中
df1 = df1.merge(patient_bmi, on='patient_id', how='left')
def separation_score(sub_df, group_col='bmi_group', target_col='pred_cross_week'):
    d = sub_df.dropna(subset=[group_col, target_col])
    if d.empty or d[target_col].nunique() < 2: return 0.0
    var_total = np.var(d[target_col], ddof=1)
    if var_total < 1e-9: return 0.0
    groups = d.groupby(group_col)
    if groups.ngroups < 2: return 0.0
    within_num = 0.0; denom = 0
    for _, g in groups:
        if g[target_col].nunique() < 2: continue
        n = len(g);
        if n <= 1: continue
        within_num += (n - 1) * np.var(g[target_col], ddof=1)
        denom += (n - 1)
    if denom == 0: return 0.0
    var_within = within_num / denom
    var_between = max(var_total - var_within, 0.0)
    return float(var_between / var_total)

def make_bin_edges(values, n_bins):
    vmin, vmax = values.min(), values.max()
    if np.isclose(vmin, vmax):
        return np.array([vmin, vmax + 1e-6])
    _, edges = np.histogram(values, bins=n_bins)
    edges = np.unique(edges)
    if len(edges) < 2: edges = np.array([vmin, vmax + 1e-6])
    return edges

if df1.empty or df1['bmi'].nunique() < 2 or df1['pred_cross_week'].nunique() < 2:
    print('数据不足，无法分箱。')
    df1['bmi_group'] = np.nan
else:
    df1['bmi_group'] = np.nan
    work_df = df1[['bmi','pred_cross_week']].copy()
    bins_list = [3,4,5,6,7,8]
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    results = []
    for n_bins in bins_list:
        fold_scores = []
        for tr_idx, va_idx in kfold.split(work_df):
            train = work_df.iloc[tr_idx]; val = work_df.iloc[va_idx]
            edges = make_bin_edges(train['bmi'].values, n_bins)
            if len(edges) < 2: continue
            val_groups = pd.cut(val['bmi'], bins=edges, include_lowest=True, right=True, labels=False)
            val_tmp = val.copy(); val_tmp['bmi_group'] = val_groups
            fold_scores.append(separation_score(val_tmp))
        if fold_scores:
            mean_score = float(np.mean(fold_scores))
            results.append((n_bins, mean_score))
            print(f'n_bins={n_bins}: CV mean separation={mean_score:.4f}')
        else:
            print(f'n_bins={n_bins}: 无有效折, 跳过')
    if not results:
        print('无可用分箱结果。')
    else:
        results.sort(key=lambda x: (-x[1], x[0])); best_bins, best_score = results[0]
        print(f'最佳候选分箱数: {best_bins}, 分离度: {best_score:.4f}')
        final_edges = make_bin_edges(work_df['bmi'].values, best_bins).astype(float)
        final_edges[-1] = np.nextafter(final_edges[-1], final_edges[-1] + 1)
        for i in range(1, len(final_edges)):
            if final_edges[i] <= final_edges[i-1]: final_edges[i] = final_edges[i-1] + 1e-9
        df1['bmi_group'] = pd.cut(df1['bmi'], bins=final_edges, include_lowest=True, right=True, labels=False)
        MIN_BIN_SIZE = max(3, int(0.01 * len(df1)))
        changed = True
        while changed:
            changed = False
            counts = df1['bmi_group'].value_counts().sort_index()
            sparse = counts[counts < MIN_BIN_SIZE]
            if sparse.empty: break
            for g in sparse.index:
                if (g-1) in counts.index:
                    target = g - 1
                elif (g+1) in counts.index:
                    target = g + 1
                else:
                    continue
                df1.loc[df1['bmi_group'] == g, 'bmi_group'] = target
                print(f'合并稀疏箱 {g} -> {target}')
                changed = True
        unique_labels = sorted(df1['bmi_group'].dropna().unique())
        remap = {old:i for i, old in enumerate(unique_labels)}
        df1['bmi_group'] = df1['bmi_group'].map(remap)
        group_ranges = df1.groupby('bmi_group')['bmi'].agg(['min','max']).sort_index()
        contiguous_edges = [group_ranges.iloc[0]['min']]
        for _, row in group_ranges.iterrows(): contiguous_edges.append(row['max'])
        contiguous_edges[-1] = np.nextafter(contiguous_edges[-1], contiguous_edges[-1] + 1)
        print('\n连续化后最终组数:', df1['bmi_group'].nunique())
        print('连续 BMI 边界:', contiguous_edges)
        print('\n各组样本数:')
        print(df1['bmi_group'].value_counts().sort_index())
        print('\nBMI 组统计:')
        print(df1.groupby('bmi_group')['bmi'].agg(['count','min','max','mean']).round(3))
        print('\n各组预测达标周统计:')
        print(df1.groupby('bmi_group')['pred_cross_week'].agg(['count','min','max','mean']).round(3))



# 设置中文字体以正确显示图例和标题
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 绘制散点图，直接在每个孕妇的特征数据上绘图
# patient_features 包含每个孕妇的唯一记录和正确的bmi_group
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df1, x='bmi', y='pred_cross_week', hue='bmi_group', palette='Set1', s=30, alpha=0.8)
plt.title('按孕妇测量平均BMI聚类后的与预测达标周散点图')
plt.xlabel('孕妇测量平均BMI')
plt.ylabel('预测达标周')
plt.legend(title='孕妇分组')
plt.grid(True)
plt.show()

# 步骤3: 定义并运行遗传算法以找到最优检测周


# 双Logistic风险函数（保持不变，可继续调节 k1,k2 控制陡峭度）
def risk_cost_function(week, m1=12.0, m2=28.0, k1=1, k2=1):
    def _sig(x):
        if x >= 50: return 1.0
        if x <= -50: return 0.0
        return 1.0 / (1.0 + math.exp(-x))
    f1 = _sig(k1 * (week - m1))
    f2 = _sig(k2 * (week - m2))
    return float(0.5 * (f1 + f2))

K_SUCCESS = 2.0
MID_SHIFT = 0.7   
def compute_success_rates(week, preds, k=K_SUCCESS, mid_shift=MID_SHIFT):
    z = k * (week - preds - mid_shift)
    # 数值稳定处理
    z = np.clip(z, -50, 50)
    success = 1.0 / (1.0 + np.exp(-z))
    return success
Target_Cost = 0.05

def evaluate(individual, group_data):
    week = individual[0]
    if week < MIN_WEEK: week = MIN_WEEK
    elif week > MAX_WEEK: week = MAX_WEEK
    individual[0] = week
    preds = group_data['pred_cross_week'].values
    if preds.size == 0:
        return (1.0,)
    success = compute_success_rates(week, preds)
    mean_failure = float(1.0 - success.mean())
    rk = risk_cost_function(week)
    total_cost = rk * mean_failure
    deviation = abs(total_cost - Target_Cost)
    return (deviation,)

def bounded_mutation(ind, mu=0, sigma=2, indpb=0.2):
    tools.mutGaussian(ind, mu=mu, sigma=sigma, indpb=indpb)
    if ind[0] < MIN_WEEK: ind[0] = MIN_WEEK
    elif ind[0] > MAX_WEEK: ind[0] = MAX_WEEK
    return (ind,)

if 'FitnessMin' not in creator.__dict__:
    creator.create('FitnessMin', base.Fitness, weights=(-1.0,))
if 'Individual' not in creator.__dict__:
    creator.create('Individual', list, fitness=creator.FitnessMin)
toolbox = base.Toolbox()
toolbox.register('attr_float', random.uniform, MIN_WEEK, MAX_WEEK)
toolbox.register('individual', tools.initRepeat, creator.Individual, toolbox.attr_float, n=1)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)
toolbox.register('mate', tools.cxBlend, alpha=0.5)
toolbox.register('mutate', bounded_mutation, mu=0, sigma=2, indpb=0.2)
toolbox.register('select', tools.selTournament, tournsize=3)

optimal_weeks = {}
for group_id in sorted(df1['bmi_group'].dropna().unique()):
    group_id = int(group_id)
    print(f'\n--- Optimizing for BMI Group {group_id} (Logistic Success * Risk) ---')
    group_data = df1[df1['bmi_group'] == group_id]
    if 'evaluate' in toolbox.__dict__: del toolbox.evaluate
    toolbox.register('evaluate', evaluate, group_data=group_data)
    population = toolbox.population(n=60)
    algorithms.eaSimple(population, toolbox, cxpb=0.6, mutpb=0.25, ngen=200, verbose=False)
    best_individual = tools.selBest(population, k=1)[0]
    optimal_weeks[group_id] = best_individual[0]
    preds = group_data['pred_cross_week'].values
    succ = compute_success_rates(best_individual[0], preds).mean() if preds.size else np.nan
    total_cost_best = risk_cost_function(best_individual[0]) * (1 - compute_success_rates(best_individual[0], preds).mean())
    print(f'  -> best week: {best_individual[0]:.3f} | total_cost: {total_cost_best:.4f} | deviation: {abs(total_cost_best - Target_Cost):.4f}')

df1['optimal_week'] = df1['bmi_group'].map(optimal_weeks)